# 03 — Business Feature Engineering

Create interpretable checkout-time features from the leakage-safe training dataset produced by notebook `02`.

## 1. Configuration and safe train loading

The same deterministic feature contract is applied to the leakage-safe train and test partitions. Analysis remains train-only, while both transformed partitions are saved for notebook `04`.

In [1]:
from bnpl_credit_risk.data.loaders import BNPLDataLoader
from bnpl_credit_risk.data.quality import DataQualityReport
from bnpl_credit_risk.features.builder import resolve_feature_columns
from bnpl_credit_risk.pipelines.feature_engineering_pipeline import (
    FeatureEngineeringPipeline,
)
from bnpl_credit_risk.settings import get_settings, load_config

settings = get_settings()
config = load_config()

safe_directory = settings.resolve(config.features.leakage_output.directory)
train_path = (
    safe_directory
    / config.features.leakage_output.train_filename
)
test_path = safe_directory / config.features.leakage_output.test_filename
missing_files = [path for path in (train_path, test_path) if not path.exists()]
if missing_files:
    missing = ", ".join(str(path) for path in missing_files)
    raise FileNotFoundError(
        f"Missing leakage-safe file(s): {missing}. "
        "Run notebook 02_exploratory_data_analysis.ipynb completely first."
    )

loader = BNPLDataLoader(settings, config.data)
train_df = loader.load_raw(train_path)
test_df = loader.load_raw(test_path)
unexpected_leaky_columns = sorted(
    set(config.features.leaky_raw_columns).intersection(
        set(train_df.columns) | set(test_df.columns)
    )
)
if unexpected_leaky_columns:
    raise ValueError(
        f"Leakage-safe train still contains: {unexpected_leaky_columns}"
    )

print(f"Active risk scope: {config.model.risk_scope}")
print(f"Train input shape: {train_df.shape}")
print(f"Test input shape: {test_df.shape}")

2026-08-27 19:16:30.695 | INFO     | bnpl_credit_risk.data.loaders:load_raw:37 - Loaded raw dataset path=/Users/surelmanda/3-Mlops-Databricks-Projects/BNPL-Credit-Risk/data/interim/train_application.csv rows=9310 columns=13
2026-08-27 19:16:30.698 | INFO     | bnpl_credit_risk.data.loaders:load_raw:37 - Loaded raw dataset path=/Users/surelmanda/3-Mlops-Databricks-Projects/BNPL-Credit-Risk/data/interim/test_application.csv rows=1035 columns=13


Active risk scope: application_risk
Train input shape: (9310, 13)
Test input shape: (1035, 13)


## 2. Business feature catalogue

| Feature | Business meaning | Formula / rule |
|---|---|---|
| `installment_amount` | Amount due for one installment | purchase ÷ number of installments |
| `installment_burden_ratio` | Share of monthly income consumed by one installment | installment amount ÷ monthly income |
| `affordability_band` | Interpretable burden level | configured burden thresholds |
| `income_after_installment` | Income remaining after one installment | income − installment amount |
| `credit_score_band` | Interpretable bureau-score segment | configured score thresholds |
| `age_group` | Non-linear age segment | configured age thresholds |
| `installment_term` | Short/medium/long BNPL duration | mapping of 3/6/9/12 installments |
| `transaction_month_sin/cos` | Cyclical seasonality | sine/cosine encoding of month |

In this dataset, `debt_to_income_ratio` is exactly `purchase_amount / monthly_income`. No reciprocal ratio is created because it would duplicate the same signal.

## 3. Configured feature creation

In [2]:
feature_pipeline = FeatureEngineeringPipeline(
    settings=settings,
    config=config,
)
feature_result = feature_pipeline.run(train_df, test_df)

train_engineered = feature_result.train
test_engineered = feature_result.test
selected_engineered_features = list(
    feature_result.selected_engineered_features
)

display(train_engineered[selected_engineered_features].head())
print(f"Selected engineered features: {selected_engineered_features}")
print(f"Train engineered shape: {train_engineered.shape}")
print(f"Test engineered shape: {test_engineered.shape}")
print(f"Saved train: {feature_result.paths.train_path}")
print(f"Saved test: {feature_result.paths.test_path}")

2026-08-27 19:16:31.061 | INFO     | bnpl_credit_risk.data.partitioning:save_train_test_csv:119 - Saved dataset partitions train=/Users/surelmanda/3-Mlops-Databricks-Projects/BNPL-Credit-Risk/data/processed/train_features.csv test=/Users/surelmanda/3-Mlops-Databricks-Projects/BNPL-Credit-Risk/data/processed/test_features.csv


,installment_amount,installment_burden_ratio,affordability_band,income_after_installment,credit_score_band,age_group,installment_term,transaction_month_sin,transaction_month_cos
0,833.333333,0.013322,Low,61718.366667,Fair,46-60,Medium,1.000000e+00,6.123234e-17
1,833.333333,0.010799,Low,76331.536667,Fair,46-60,Medium,8.660254e-01,-5.000000e-01
2,228.720833,0.026750,Moderate,8321.459167,Poor,46-60,Extended,1.224647e-16,-1.000000e+00
3,555.555556,0.006421,Very Low,85961.494444,Very Good,46-60,Long,8.660254e-01,5.000000e-01
4,488.595556,0.016013,Low,30023.844444,Poor,18-25,Long,-5.000000e-01,8.660254e-01


Selected engineered features: ['installment_amount', 'installment_burden_ratio', 'affordability_band', 'income_after_installment', 'credit_score_band', 'age_group', 'installment_term', 'transaction_month_sin', 'transaction_month_cos']
Train engineered shape: (9310, 22)
Test engineered shape: (1035, 22)
Saved train: /Users/surelmanda/3-Mlops-Databricks-Projects/BNPL-Credit-Risk/data/processed/train_features.csv
Saved test: /Users/surelmanda/3-Mlops-Databricks-Projects/BNPL-Credit-Risk/data/processed/test_features.csv


## 4. Engineered feature quality

Check missing values, cardinality, infinities, outliers and representative values before these columns reach preprocessing.

In [3]:
feature_profile = DataQualityReport().profile_dataset(
    train_engineered[selected_engineered_features]
)
display(feature_profile)

2026-08-27 19:16:31.077 | INFO     | bnpl_credit_risk.data.profiling:inspect:56 - Computing dataset quality summary rows=9310 columns=9


,Column,Type,Non-Null,Missing,% Missing,Cardinality,% Unique,Zeros,% Zeros,Infinite,Outliers,% Outliers,Min,Mean,Median,Max,Std Dev,Examples
0,installment_amount,float64,9310,0,0.0,3617,38.85,0,0.0,0,1460,15.68,8.3333,689.3136,555.5556,1666.6667,487.629,"[833.3333333333334, 228.72083333333333, 555.55..."
1,installment_burden_ratio,float64,9310,0,0.0,9298,99.87,0,0.0,0,689,7.4,0.0001,0.0314,0.0221,0.2412,0.0303,"[0.013322313115923842, 0.010799387510577461, 0..."
2,affordability_band,object,9310,0,0.0,5,0.05,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"[Low, Moderate, Very Low, Very High, High]"
3,income_after_installment,float64,9310,0,0.0,9297,99.86,0,0.0,0,22,0.24,3974.58,34313.5275,23014.0304,145350.4533,26961.6237,"[61718.36666666666, 76331.53666666667, 8321.45..."
4,credit_score_band,object,9310,0,0.0,5,0.05,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"[Fair, Poor, Very Good, Good, Excellent]"
5,age_group,object,9310,0,0.0,4,0.04,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"[46-60, 18-25, 36-45, 26-35]"
6,installment_term,object,9310,0,0.0,4,0.04,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"[Medium, Extended, Long, Short]"
7,transaction_month_sin,float64,9310,0,0.0,11,0.12,0,0.0,0,0,0.0,-1.0,-0.0071,-0.0,1.0,0.7072,"[1.0, 0.8660254037844387, 1.2246467991473532e-..."
8,transaction_month_cos,float64,9310,0,0.0,11,0.12,0,0.0,0,0,0.0,-1.0,0.0047,0.0,1.0,0.7071,"[6.123233995736766e-17, -0.49999999999999983, ..."


## 5. Assumptions and responsible use

- Installments are assumed to be equal and fee-free because the dataset contains no interest or fee field.
- Comparing one installment with monthly income assumes the repayment cadence is compatible with a monthly affordability view.
- `age` and `employment_type` may require fairness and regulatory review before a real credit decision deployment.
- Thresholds are configuration, not universal credit policy; they must be validated on future out-of-time data.

## 6. Feature scopes and leakage boundary

The production application scope contains only checkout-time features. Behavioral additions remain reserved for post-origination monitoring.

In [4]:
application_numeric, application_categorical = resolve_feature_columns(
    config.features,
    "application_risk",
)
behavioral_numeric, behavioral_categorical = resolve_feature_columns(
    config.features,
    "behavioral_risk",
)

behavioral_only_numeric = sorted(
    set(behavioral_numeric) - set(application_numeric)
)
behavioral_only_categorical = sorted(
    set(behavioral_categorical) - set(application_categorical)
)

print(f"Application numeric: {application_numeric}")
print(f"Application categorical: {application_categorical}")
print(f"Behavioral-only numeric: {behavioral_only_numeric}")
print(f"Behavioral-only categorical: {behavioral_only_categorical}")

Application numeric: ['age', 'monthly_income', 'credit_score', 'purchase_amount', 'bnpl_installments', 'app_usage_frequency', 'debt_to_income_ratio', 'installment_amount', 'installment_burden_ratio', 'income_after_installment', 'transaction_month_sin', 'transaction_month_cos']
Application categorical: ['employment_type', 'product_category', 'location', 'age_group', 'credit_score_band', 'affordability_band', 'installment_term']
Behavioral-only numeric: ['is_high_risk', 'missed_payments', 'payment_stress', 'repayment_delay_days', 'risk_score']
Behavioral-only categorical: ['customer_segment']


### Output for the next step

Notebook `04` loads `data/processed/train_features.csv` and `data/processed/test_features.csv`. The persisted model pipeline recomputes deterministic features internally so future raw checkout inputs follow the exact same transformations.